In [1]:

import pandas as pd 
import numpy as np 
import os
import sys
sys.path.append('../')
sys.path.append('../..')
import random

from detection.lof import LocalOutlierFactorDetector
from sklearn.metrics import classification_report, confusion_matrix \
        , roc_auc_score, f1_score, accuracy_score, precision_score, recall_score

from preprocessing.load_data import DataLoader

In [2]:


card_data = pd.read_csv('../data/raw/creditcard.csv')

print("Card data shape: ", card_data.shape) 

print(card_data.head(2))


data_use = DataLoader(data=card_data, label_col='Class')

data_use.remove_columns(['Time'])

Card data shape:  (284807, 31)
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   

        V26       V27       V28  Amount  Class  
0 -0.189115  0.133558 -0.021053  149.62      0  
1  0.125895 -0.008983  0.014724    2.69      0  

[2 rows x 31 columns]


In [3]:

training, label = data_use.get_features(), data_use.get_labels()

print("Training data shape: ", training.shape) 

print(training.head(2))

print("label data shape: ", label.shape) 

print(label.head(2))

X_train, X_test, y_train, y_test = data_use.train_test_split(test_size=0.3, random_state=42)

print("X_train shape: ", X_train.shape) 
print("X_test shape: ", X_test.shape)

print("Sample X_train data: ", X_train.head(2))

Training data shape:  (284807, 29)
         V1        V2        V3        V4        V5        V6        V7  \
0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   

         V8        V9       V10  ...       V20       V21       V22       V23  \
0  0.098698  0.363787  0.090794  ...  0.251412 -0.018307  0.277838 -0.110474   
1  0.085102 -0.255425 -0.166974  ... -0.069083 -0.225775 -0.638672  0.101288   

        V24       V25       V26       V27       V28  Amount  
0  0.066928  0.128539 -0.189115  0.133558 -0.021053  149.62  
1 -0.339846  0.167170  0.125895 -0.008983  0.014724    2.69  

[2 rows x 29 columns]
label data shape:  (284807,)
0    0
1    0
Name: Class, dtype: int64
X_train shape:  (199364, 29)
X_test shape:  (85443, 29)
Sample X_train data:                V1        V2        V3        V4        V5        V6        V7  \
2557   -2.289565 -0.480260  0.818685 -1.706423  0.822102 -1.66

## Special

In [4]:
# Due to autoencoder being an unsupervised method
# we will train only on the normal transactions (label 0) and test on the full test set
X_train_normal = X_train[y_train == 0]
print(X_train_normal.shape)
X_train_normal.head(2) 

(199008, 29)


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
2557,-2.289565,-0.480260,0.818685,-1.706423,0.822102,-1.660326,0.944047,-0.541765,1.323156,-0.434426,...,-0.831985,-0.210837,0.914737,0.867888,0.422969,0.310584,-0.781488,0.392241,-0.147757,1.00
247823,-0.313717,-4.064342,-3.398445,0.704011,0.101662,1.529848,1.551670,-0.036774,0.015829,-0.359561,...,2.142593,0.853186,-0.091941,-0.936215,-0.833081,-0.498728,0.651183,-0.290331,0.110360,1194.28


In [ ]:

from detection.autoencoder import AutoencoderDetector


print("Start training Autoencoder model...")
autoencoder = AutoencoderDetector().build(input_shape=(X_train_normal.shape[1],))

autoencoder.fit(X_train_normal, epochs=10, batch_size=16)

print("Model training completed.")

print("Scoring test data...")

autoencoder_scores = autoencoder.score(X_test)

autoencoder_predictions = autoencoder.predict(X_test)

print("Autoencoder Scores: ", autoencoder_scores[:5])
print("Autoencoder Predictions: ", autoencoder_predictions[:5])


Start training Autoencoder model...
Epoch 1/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step - loss: 93.2131 - val_loss: 0.6350
Epoch 2/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - loss: 2.3308 - val_loss: 0.3917
Epoch 3/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 1.5114 - val_loss: 0.6003
Epoch 4/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 33s 3ms/step - loss: 2.4794 - val_loss: 0.5559
Epoch 5/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: 1.4013 - val_loss: 0.8654
Epoch 6/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step - loss: 1.7784 - val_loss: 0.8160
Epoch 7/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 46s 3ms/step - loss: 1.6379 - val_loss: 0.9720
Epoch 8/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 29s 3ms/step - loss: 0.9785 - val_loss: 0.2791
Epoch 9/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - loss: 1.2552 - val_loss: 1.8297
Epoch 10/10
11195/11195 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - loss: 0.7472 - val_loss: 0.3594
Model training completed.
Scoring test data...
2671/

In [8]:

print("Evaluating model performance...")

if y_test is not None:
    print("Number of anomalies detected: ", np.sum(autoencoder_predictions))
    print("Percentage of anomalies detected: ", np.mean(autoencoder_predictions) * 100, "%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, autoencoder_predictions.astype(int)))
    print("\nClassification Report:")
    print(classification_report(y_test, autoencoder_predictions.astype(int)))

    print("AUC-ROC of autoencoder: ", roc_auc_score(y_test, autoencoder_scores))

    if not os.path.exists('../models/checkpoint/'):
        os.makedirs('../models/checkpoint/')

    result_folder = '../models/checkpoint/baseline_autoencoder/'
    if not os.path.exists(result_folder):
        os.makedirs(result_folder)

    autoencoder.save_model(result_folder + 'autoencoder_baseline_model.keras')

    classification_report_df = classification_report(y_test, autoencoder_predictions.astype(int), output_dict=True)

    report_df = pd.DataFrame(classification_report_df).transpose()
    report_df.to_csv(result_folder + 'autoencoder_classification_baseline_report.csv', index=True)

else:
    print("No labels available for evaluation.")
    print("Number of detected anomalies: ", np.sum(autoencoder_predictions))

Evaluating model performance...
Number of anomalies detected:  354
Percentage of anomalies detected:  0.4143112952494646 %
Confusion Matrix:
[[85034   273]
 [   55    81]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85307
           1       0.23      0.60      0.33       136

    accuracy                           1.00     85443
   macro avg       0.61      0.80      0.66     85443
weighted avg       1.00      1.00      1.00     85443

AUC-ROC of autoencoder:  0.9491357856985738
